In [3]:
import pandas as pd
import numpy as np
import igraph as ig
import re
import os

In [18]:
network = pd.read_csv('/home/lnemati/pathway_crosstalk/results/crosstalk/all_ccc_complex_pairs/adj/network_filtered.csv')
network = network.query('auroc > 0.5')
network = network[~network['ccc']]

In [19]:
import pandas as pd
import igraph as ig

# assume your dataframe is called `network`

# 1. Build edge list
edges = list(zip(network["complex1"], network["complex2"]))

# 2. Get unique node names
nodes = list(set(network["complex1"]).union(set(network["complex2"])))

# 3. Create graph
g = ig.Graph()
g.add_vertices(nodes)
g.add_edges(edges)

# Ensure it's undirected and unweighted
g = g.as_undirected()
g.simplify()  # removes duplicates / self-loops if any

In [20]:
# 4. Find cliques
cliques_3 = g.cliques(min=3, max=3)
cliques_4 = g.cliques(min=4, max=4)

# 5. Convert vertex indices back to names
cliques_3_named = [[g.vs[idx]["name"] for idx in clique] for clique in cliques_3]
cliques_4_named = [[g.vs[idx]["name"] for idx in clique] for clique in cliques_4]

print("3-node cliques:", cliques_3_named[:5])
print("4-node cliques:", cliques_4_named[:5])

3-node cliques: [['ACVR2A_BMPR1A', 'CD24', 'ACVR1_BMPR2'], ['ACVR2A_BMPR1A', 'IL20RA_IL20RB_IL2RG', 'ACVR1_BMPR2'], ['ACVR2A_BMPR1A', 'ERBB3', 'ACVR1_BMPR2'], ['ACVR2A_BMPR1A', 'DAGLA', 'ACVR1_BMPR2'], ['ACVR2A_BMPR1A', 'ACVR1_BMPR2', 'CEL']]
4-node cliques: [['ACVR1_ACVR2A', 'CD8A_CD8B', 'PRLR', 'ENPP4'], ['ACVR1_ACVR2A', 'IL15RA_IL2RB_IL2RG', 'CD8A_CD8B', 'PRLR'], ['ACVR1_ACVR2A', 'IL21R_IL2RG', 'CD8A_CD8B', 'PRLR'], ['ACVR1_ACVR2A', 'CD8A_CD8B', 'IL2RA_IL2RB_IL2RG', 'PRLR'], ['CD8A_CD8B', 'ACVR2A_BMPR1A', 'IL2RA_IL2RB_IL2RG', 'EFNA5']]


In [47]:
487 + 3818

4305

In [48]:
487 / 3818

0.12755369303300157

In [39]:
import random
import pandas as pd

N3 = 487
N4 = 3818
factor = 10

# sample cliques
sample_3 = random.sample(cliques_3_named, min(len(cliques_3_named), factor * N3))
sample_4 = random.sample(cliques_4_named, min(len(cliques_4_named), factor * N4))

# --- 3-node cliques ---
interactions = []
for clique in sample_3:
    shared = random.choice(clique)
    others = [x for x in clique if x != shared]
    int1 = f"{shared}+{others[0]}"
    int2 = f"{shared}+{others[1]}"
    interactions.append(f"{int1}&{int2}")

df3 = pd.DataFrame(interactions, columns=["Interaction"])
df3["Type"] = "3_non_cci_clique"


# --- 4-node cliques ---
interactions = []
for clique in sample_4:
    pair1 = random.sample(clique, 2)
    pair2 = [x for x in clique if x not in pair1]
    int1 = f"{pair1[0]}+{pair1[1]}"
    int2 = f"{pair2[0]}+{pair2[1]}"
    interactions.append(f"{int1}&{int2}")

df4 = pd.DataFrame(interactions, columns=["Interaction"])
df4["Type"] = "4_non_cci_clique"

In [42]:
non_cci_cliques = pd.concat([df3, df4])

In [44]:
non_cci_cliques.to_csv('/home/lnemati/pathway_crosstalk/results/crosstalk/all_ccc_complex_pairs/adj/non_cci_cliques/motifs.csv')

In [4]:
non_cci_cliques = pd.read_csv('/home/lnemati/pathway_crosstalk/results/crosstalk/all_ccc_complex_pairs/adj/non_cci_cliques/motifs.csv', index_col=0)
non_cci_cliques

,Interaction,Type
0,COL22A1+NRP1_PLXNA1&COL22A1+STRA6,3_non_cci_clique
1,ULBP3+CD8A_CD8B&ULBP3+ITGA4_ITGB7,3_non_cci_clique
2,TNFSF4+CD200R1&TNFSF4+CD28,3_non_cci_clique
3,HRH1+NECTIN4&HRH1+PTHLH,3_non_cci_clique
4,F2RL2+IL24&F2RL2+ENPP1,3_non_cci_clique
...,...,...
38175,LILRB3+CD28&CD2+ADA2,4_non_cci_clique
38176,ITGAX_ITGB2+FPR1&TNFSF10+TREM2_TYROBP,4_non_cci_clique
38177,SIGLEC10+P2RY6&CD48+COL12A1,4_non_cci_clique
38178,TEK+ACVR1_ACVR2A&POSTN+THY1,4_non_cci_clique


In [5]:
non_cci_cliques_reduced = non_cci_cliques.iloc[::5]
non_cci_cliques_reduced.to_csv('/home/lnemati/pathway_crosstalk/results/crosstalk/all_ccc_complex_pairs/adj/non_cci_cliques/motifs_reduced_for_immunotherapy.csv')

In [8]:
pseudo_ccc = pd.DataFrame(list(set(non_cci_cliques_reduced.Interaction.str.split('&').sum())), columns=['interaction'])
pseudo_ccc['complex_a'] = pseudo_ccc['interaction'].str.split('+', expand=True)[0]
pseudo_ccc['complex_b'] = pseudo_ccc['interaction'].str.split('+', expand=True)[1]
pseudo_ccc['all_genes'] = pseudo_ccc['interaction'].apply(lambda x: sorted(re.split('[+_]', x)))
keep = pseudo_ccc['all_genes'].drop_duplicates().index
pseudo_ccc = pseudo_ccc.loc[keep].reset_index()
pseudo_ccc.to_csv('/home/lnemati/pathway_crosstalk/data/interactions/pseudo_ccc_reduced_from_non_cci_cliques.csv')

In [7]:
pseudo_ccc

,index,interaction,complex_a,complex_b,all_genes
0,0,LCK+TNFSF8,LCK,TNFSF8,"[LCK, TNFSF8]"
1,1,ANGPTL4+COL7A1,ANGPTL4,COL7A1,"[ANGPTL4, COL7A1]"
2,2,IL15RA_IL2RB_IL2RG+UBASH3B,IL15RA_IL2RB_IL2RG,UBASH3B,"[IL15RA, IL2RB, IL2RG, UBASH3B]"
3,3,PILRA+P2RY6,PILRA,P2RY6,"[P2RY6, PILRA]"
4,4,SEMA3G+CYSLTR1,SEMA3G,CYSLTR1,"[CYSLTR1, SEMA3G]"
...,...,...,...,...,...
13134,14484,CXCR4+LILRB1,CXCR4,LILRB1,"[CXCR4, LILRB1]"
13135,14486,ITGA4_ITGB1+P2RY13,ITGA4_ITGB1,P2RY13,"[ITGA4, ITGB1, P2RY13]"
13136,14487,HCST_KLRK1+CD72,HCST_KLRK1,CD72,"[CD72, HCST, KLRK1]"
13137,14489,CDH11+TNFSF8,CDH11,TNFSF8,"[CDH11, TNFSF8]"


In [30]:
pseudo_ccc

,index,interaction,complex_a,complex_b,all_genes
0,0,NT5E_SLC29A1+CXCL10,NT5E_SLC29A1,CXCL10,"[CXCL10, NT5E, SLC29A1]"
1,1,TNFSF4+GPR34,TNFSF4,GPR34,"[GPR34, TNFSF4]"
2,2,CXCR6+TREM2_TYROBP,CXCR6,TREM2_TYROBP,"[CXCR6, TREM2, TYROBP]"
3,3,SIRPG+BOC_PTCH1,SIRPG,BOC_PTCH1,"[BOC, PTCH1, SIRPG]"
4,4,S1PR1+SIRPG,S1PR1,SIRPG,"[S1PR1, SIRPG]"
...,...,...,...,...,...
34005,45879,TNFRSF18+TNFSF4,TNFRSF18,TNFSF4,"[TNFRSF18, TNFSF4]"
34006,45880,ENPP1+ACVR2A_BMPR1A,ENPP1,ACVR2A_BMPR1A,"[ACVR2A, BMPR1A, ENPP1]"
34007,45882,IL1B+FLT3LG,IL1B,FLT3LG,"[FLT3LG, IL1B]"
34008,45883,EPHA4+CCR7,EPHA4,CCR7,"[CCR7, EPHA4]"


# Immunotherapy

In [23]:
import os
import pandas as pd
from scipy.stats import fisher_exact

# =========================
# Helper to load + filter
# =========================
def load_dataset(parentdir, motif_filter):
    dfs = []

    for tissue in os.listdir(parentdir):
        path = os.path.join(parentdir, tissue, 'aggregated', 'aggregated.csv')

        if not os.path.exists(path):
            continue

        tissuedf = pd.read_csv(path)
        tissuedf = tissuedf.rename(columns={'Unnamed: 0': 'interaction'})
        dfs.append(tissuedf)

    df = pd.concat(dfs, ignore_index=True)

    df = df.sort_values(by='auroc', ascending=False)
    df = df.query('motif != "random_pairs"')
    df = df.query(f'motif in {motif_filter}')

    return df


# =========================
# Load both datasets
# =========================
cci_parent = '/home/lnemati/pathway_crosstalk/results/immunotherapy/individual_interactions_and_motifs'
noncci_parent = '/home/lnemati/pathway_crosstalk/results/immunotherapy/non_cci_cliques/individual_interactions_and_motifs'

cci_df = load_dataset(cci_parent, ["3_clique", "4_clique"])
noncci_df = load_dataset(noncci_parent, ["3_non_cci_clique", "4_non_cci_clique"])


# =========================
# Function to compute stats
# =========================
def compute_fraction(df, condition):
    den = df.shape[0]
    num = df.query(condition).shape[0]
    return num, den, num / den if den > 0 else float('nan')


def compute_counts(df, condition):
    better = df.query(condition).shape[0]
    worse = df.shape[0] - better
    return better, worse


# =========================
# Conditions
# =========================
conds = {
    "AUROC": "auroc > auroc1 and auroc > auroc2",
    "AUPRC": "auprc > auprc1 and auprc > auprc2",
    "BOTH":  "auroc > auroc1 and auroc > auroc2 and auprc > auprc1 and auprc > auprc2"
}


# =========================
# Print original fractions
# =========================
print("=== FRACTIONS ===")

for name, cond in conds.items():
    print(f"\n{name}")

    num, den, frac = compute_fraction(cci_df, cond)
    print(f"CCI:     {num}/{den} = {frac:.4f}")

    num, den, frac = compute_fraction(noncci_df, cond)
    print(f"non-CCI: {num}/{den} = {frac:.4f}")


# =========================
# Fisher exact tests
# =========================
print("\n=== FISHER EXACT TESTS ===")

for name, cond in conds.items():
    a, b = compute_counts(cci_df, cond)
    c, d = compute_counts(noncci_df, cond)

    table = [[a, b], [c, d]]
    oddsratio, pvalue = fisher_exact(table)

    print(f"\n{name}")
    print(f"Contingency table: [[CCI better={a}, worse={b}], [nonCCI better={c}, worse={d}]]")
    print(f"Odds ratio: {oddsratio:.4f}")
    print(f"P-value: {pvalue:.4e}")

=== FRACTIONS ===

AUROC
CCI:     13399/38745 = 0.3458
non-CCI: 21171/69854 = 0.3031

AUPRC
CCI:     13840/38745 = 0.3572
non-CCI: 22180/69854 = 0.3175

BOTH
CCI:     10534/38745 = 0.2719
non-CCI: 16360/69854 = 0.2342

=== FISHER EXACT TESTS ===

AUROC
Contingency table: [[CCI better=13399, worse=25346], [nonCCI better=21171, worse=48683]]
Odds ratio: 1.2156
P-value: 2.9333e-47

AUPRC
Contingency table: [[CCI better=13840, worse=24905], [nonCCI better=22180, worse=47674]]
Odds ratio: 1.1945
P-value: 3.4933e-40

BOTH
Contingency table: [[CCI better=10534, worse=28211], [nonCCI better=16360, worse=53494]]
Odds ratio: 1.2209
P-value: 7.7965e-43


In [1]:
import pandas as pd
import numpy as np

In [4]:
# CCIs

## Read the results
parentdir = '/home/lnemati/pathway_crosstalk/results/immunotherapy/individual_interactions_and_motifs'

dfs = []

for tissue in os.listdir(parentdir):
    #if tissue in ['full_dataset']: # add back lymph and ureter when they finish running
    #    continue
    path = os.path.join(parentdir, tissue, 'aggregated', 'aggregated.csv')
    tissuedf = pd.read_csv(os.path.join(path))
    tissuedf = tissuedf.rename(columns={'Unnamed: 0': 'interaction'})
    #tissuedf['motif'] = tissuedf['motif'].replace({'3_clique': 'cliques', '4_clique': 'cliques'})

    dfs.append(tissuedf)
        
crosstalkdf = pd.concat(dfs, ignore_index=True)
#crosstalkdf = crosstalkdf.dropna()

crosstalkdf = crosstalkdf.sort_values(by='auroc')[::-1]

crosstalkdf = crosstalkdf.query('motif != "random_pairs"')
#crosstalkdf = crosstalkdf.query('tissue != "full_dataset"')

better = crosstalkdf.query('auroc > auroc1 and auroc > auroc2')
better = better.query('auprc > auprc1 and auprc > auprc2')

crosstalkdf = crosstalkdf.query('motif in ["3_clique", "4_clique"]')
den = crosstalkdf.shape[0]

print('AUROC')

num = crosstalkdf.query('auroc > auroc1 and auroc > auroc2').shape[0]
print(num / den)

print('AUPRC')

num = crosstalkdf.query('auprc > auprc1 and auprc > auprc2').shape[0]
print(num / den)

print('BOTH')

num = crosstalkdf.query('auroc > auroc1 and auroc > auroc2 and auprc > auprc1 and auprc > auprc2').shape[0]
print(num / den)


In [19]:
# NON-CCIs

## Read the results
parentdir = '/home/lnemati/pathway_crosstalk/results/immunotherapy/non_cci_cliques/individual_interactions_and_motifs'

dfs = []

for tissue in os.listdir(parentdir):
    #if tissue in ['full_dataset']: # add back lymph and ureter when they finish running
    #    continue
    path = os.path.join(parentdir, tissue, 'aggregated', 'aggregated.csv')
    if not os.path.exists(os.path.join(path)):
        continue
    tissuedf = pd.read_csv(os.path.join(path))
    tissuedf = tissuedf.rename(columns={'Unnamed: 0': 'interaction'})
    #tissuedf['motif'] = tissuedf['motif'].replace({'3_clique': 'cliques', '4_clique': 'cliques'})

    dfs.append(tissuedf)
        
crosstalkdf = pd.concat(dfs, ignore_index=True)
#crosstalkdf = crosstalkdf.dropna()

crosstalkdf = crosstalkdf.sort_values(by='auroc')[::-1]

crosstalkdf = crosstalkdf.query('motif != "random_pairs"')
#crosstalkdf = crosstalkdf.query('tissue != "full_dataset"')

better = crosstalkdf.query('auroc > auroc1 and auroc > auroc2')
better = better.query('auprc > auprc1 and auprc > auprc2')

crosstalkdf = crosstalkdf.query('motif in ["3_non_cci_clique", "4_non_cci_clique"]')
den = crosstalkdf.shape[0]

print('AUROC')

num = crosstalkdf.query('auroc > auroc1 and auroc > auroc2').shape[0]
print(num / den)

print('AUPRC')

num = crosstalkdf.query('auprc > auprc1 and auprc > auprc2').shape[0]
print(num / den)

print('BOTH')

num = crosstalkdf.query('auroc > auroc1 and auroc > auroc2 and auprc > auprc1 and auprc > auprc2').shape[0]
print(num / den)


AUROC
0.3030749849686489
AUPRC
0.3175193976007101
BOTH
0.23420276576860308
